# Intent Classification Model Fine-tuning (Improved)

This notebook fine-tunes a transformer-based classification model for intent detection with improvements for imbalanced data.

**Improvements**:
- Class weighting for imbalanced data
- Focal Loss for better minority class handling
- Data augmentation for underrepresented classes
- Better model selection (BERT-base)
- Improved hyperparameters

**Task**: Multi-class classification (14 intent classes)  
**Model**: distilbert-base-uncased (still lightweight)  
**Data**: Bilingual (English + Roman Urdu) booking assistant messages

---

## 1. Setup & Dependencies

In [55]:
# Install required packages (run once)
# !pip install transformers datasets scikit-learn pandas torch accelerate

In [56]:
!pip install groq

In [57]:
import re
import json
import os
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter
import random

from groq import Groq

from sklearn.model_selection import train_test_split

# Set random seeds for reproducibility
SEED = 67
random.seed(SEED)
np.random.seed(SEED)

## 2. Configuration

In [58]:
# Paths
DATASET_PATH = "/kaggle/input/datasets/taqikaaccount/intent-dataset-merged/intent_dataset_merged.jsonl"
# DATASET_PATH = "/kaggle/input/datasets/taqikaaccount/intent-dataset-merged/intent_dataset_merged.jsonl"

# Model configuration (Groq)
MODEL_NAME = "qwen/qwen3-32b"  # update if Groq uses a different id

CONFIG = {
    "max_length": 128,
}

# 5 CORE CLASSES (The New Taxonomy)
INTENT_LABELS = [
    "greeting",
    "inquiry",
    "info",
    "transaction_confirm",
    "unknown"
]

# MAPPING FROM 14 CLASSES TO 5 CLASSES (The Critical Logic)
INTENT_MAPPING = {
    "greeting": "greeting",
    "booking_request": "inquiry",
    "availability_inquiry": "inquiry",
    "service_selection": "inquiry",
    "date_selection": "inquiry",
    "time_selection": "inquiry",
    "price_inquiry": "info",
    "information": "info",
    "payment_related": "info",
    "confirmation": "transaction_confirm",
    "cancellation": "transaction_confirm",
    "modification": "transaction_confirm",
    "name_provided": "unknown",
    "unknown": "unknown"
}

# Create label mappings
label2id = {label: idx for idx, label in enumerate(INTENT_LABELS)}
id2label = {idx: label for idx, label in enumerate(INTENT_LABELS)}

# Replace with your actual API key or use environment variable
GROQ_API_KEY = "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"
if not GROQ_API_KEY:
    raise ValueError("Missing GROQ_API_KEY. Set it as an environment variable.")

client = Groq(api_key=GROQ_API_KEY)

print(f"Number of classes: {len(INTENT_LABELS)}")
print(f"Model (Groq): {MODEL_NAME}")

Number of classes: 5
Model (Groq): qwen/qwen3-32b


## 3. Custom Loss Functions

In [59]:
pass

## 4. Load & Explore Data

In [60]:
def load_jsonl(filepath):
    """Load JSONL file into list of dictionaries."""
    data = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

# Load the merged dataset
raw_data = load_jsonl(DATASET_PATH)
print(f"Total samples loaded: {len(raw_data)}")

# Convert to DataFrame for easier exploration
df = pd.DataFrame(raw_data)

# Handle chat format if needed
if 'messages' in df.columns:
    print("Detected chat format, flattening...")
    df['text'] = df['messages'].apply(lambda x: x[1]['content'])  # User message
    df['intent'] = df['messages'].apply(lambda x: x[2]['content'])  # Assistant = intent
    df = df.drop(columns=['messages'])

# === NEW LOGIC: APPLY THE 5-CLASS MAPPING ===
print(f"\nApplying class reduction mapping...")
df['original_intent'] = df['intent'] # Keep original for reference
# Map the 14 intents to the 5 new ones
df['intent'] = df['intent'].map(lambda x: INTENT_MAPPING.get(x, "unknown"))

df.head(10)

Total samples loaded: 694

Applying class reduction mapping...


,text,intent,original_intent
0,Hi,greeting,greeting
1,Hello,greeting,greeting
2,Hey,greeting,greeting
3,Aoa,greeting,greeting
4,Salam,greeting,greeting
5,Salaam,greeting,greeting
6,Assalam o Alaikum,greeting,greeting
7,As-salamu alaykum,greeting,greeting
8,Assalamualaikum,greeting,greeting
9,walaikum assalam,greeting,greeting


In [61]:
# Dataset statistics
print("=" * 50)
print("DATASET STATISTICS")
print("=" * 50)

print(f"\nTotal samples: {len(df)}")
print(f"Unique intents: {df['intent'].nunique()}")

# Intent distribution
intent_counts = df['intent'].value_counts()
print("\nIntent Distribution:")
print("-" * 40)
for intent, count in intent_counts.items():
    pct = (count / len(df)) * 100
    bar = "█" * int(pct / 2)
    print(f"{intent:22s}: {count:4d} ({pct:5.1f}%) {bar}")

DATASET STATISTICS

Total samples: 694
Unique intents: 5

Intent Distribution:
----------------------------------------
inquiry               :  277 ( 39.9%) ███████████████████
info                  :  171 ( 24.6%) ████████████
transaction_confirm   :  147 ( 21.2%) ██████████
unknown               :   62 (  8.9%) ████
greeting              :   37 (  5.3%) ██


In [62]:
def create_fewshot_prompt(text):
    examples = """
User message: Hi there!
Intent: greeting

User message: I'd like to book a table for two tomorrow at 7pm.
Intent: inquiry

User message: What are your prices for the deluxe package?
Intent: info

User message: Please confirm my booking #12345.
Intent: transaction_confirm

User message: My name is John.
Intent: unknown
"""
    return f"{examples}\nUser message: {text}\nIntent:"


def classify_intent(text, use_fewshot=True):
    try:
        if use_fewshot:
            user_content = create_fewshot_prompt(text)
            messages = [{"role": "user", "content": user_content}]
        else:
            system_message = """You are an intent classifier for a booking system. Your task is to classify user messages into exactly one of the following intents:

- greeting: Salutations, hellos, hi, good morning, etc.
- inquiry: Questions about booking, availability, dates, times, services.
- info: Requests for general information (prices, payment, details) not tied to a booking.
- transaction_confirm: Messages confirming, modifying, or canceling a booking.
- unknown: Anything that doesn't fit the above, including name providing, small talk, etc.

Respond with ONLY the intent label, nothing else. Do not add punctuation or explanation."""
            user_message = f"User message: {text}\nIntent:"
            messages = [
                {"role": "system", "content": system_message},
                {"role": "user", "content": user_message},
            ]

        completion = client.chat.completions.create(
            model=MODEL_NAME,
            messages=messages,
            temperature=0.0,
            max_tokens=2048,   # Must be large enough for <think> block to finish
        )

        raw = completion.choices[0].message.content or ""
        # ── KEY FIX: Qwen3 is a reasoning model that wraps every reply in
        #    <think>...</think> before the actual answer.  With max_tokens=10
        #    (the original value) the block never closes and we only ever see
        #    the opening tag → everything maps to "unknown".
        #    Strip it first, then grab the real label.
        cleaned = re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL)
        response = cleaned.strip().lower()
        first_token = response.split()[0] if response.split() else ""

        if first_token in INTENT_LABELS:
            return first_token
        if response in INTENT_LABELS:
            return response

        print(f"Warning: Unexpected response: '{response[:80]}', mapping to unknown")
        return "unknown"

    except Exception as e:
        print(f"Error processing '{text}': {e}")
        return "unknown"

## 5. Data Augmentation

In [63]:
# Optional: use nlpaug for richer text augmentation if available
try:
    import nlpaug.augmenter.word as naw
    _HAS_NLPAUG = True
    _SYNONYM_AUG = naw.SynonymAug(aug_src='wordnet')
except Exception:
    _HAS_NLPAUG = False
    _SYNONYM_AUG = None


def augment_minority_classes(
    df,
    min_samples=20,
    target_intents=None,
    use_library=True,
):
    """Augment under‑represented classes in a DataFrame.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame with at least `text` and `intent` columns.
    min_samples : int, optional
        Ensure each selected intent has at least this many samples.
    target_intents : list[str] | None, optional
        If provided, only these intents are considered for augmentation.
        If None, all intents in INTENT_LABELS are considered.
    use_library : bool, optional
        If True and `nlpaug` is installed, use a synonym augmenter in
        addition to the simple rule‑based augmentations.

    Returns
    -------
    pd.DataFrame
        Original DataFrame with augmented rows appended.

    Notes
    -----
    You can call this function manually to boost specific classes, e.g.::

        df_train = augment_minority_classes(
            df_train,
            min_samples=80,
            target_intents=["greeting", "unknown"],
        )
    """
    augmented_rows = []

    intents_to_process = target_intents if target_intents is not None else INTENT_LABELS

    for intent in intents_to_process:
        intent_df = df[df['intent'] == intent]
        count = len(intent_df)

        if count >= min_samples or intent_df.empty:
            continue

        samples_needed = min_samples - count
        print(f"Augmenting '{intent}': {count} -> {min_samples} samples")

        for _ in range(samples_needed):
            sample = intent_df.sample(1).iloc[0]
            text = sample['text']
            augmented_text = text

            # 1) Library-based synonym augmentation (if available)
            if use_library and _HAS_NLPAUG and random.random() < 0.5:
                try:
                    augmented_text = _SYNONYM_AUG.augment(text)
                    # nlpaug can return list or str depending on version
                    if isinstance(augmented_text, list) and augmented_text:
                        augmented_text = augmented_text[0]
                except Exception:
                    augmented_text = text

            # 2) Simple augmentation techniques for Roman Urdu + English
            if augmented_text == text:
                r = random.random()

                # 2.a Add polite markers (common in Urdu/English mix)
                if r < 0.3:
                    words = text.split()
                    if random.random() < 0.5:
                        augmented_text = " ".join(words + ["ji"])
                    else:
                        augmented_text = " ".join(["bhai"] + words)

                # 2.b Case variation
                elif r < 0.5:
                    augmented_text = text.lower()

                # 2.c Add punctuation variations
                elif r < 0.7:
                    if not text.endswith('?'):
                        augmented_text = text + "?"
                    elif not text.endswith('.'):
                        augmented_text = text + "."

                # 2.d Simple token replacements / expansions
                else:
                    replacements = {
                        "book": "booking",
                        "slot": "time slot",
                        "tmrw": "tomorrow",
                        "tmr": "tomorrow",
                        "kal": "tomorrow",
                        "aaj": "today",
                        "shaam": "evening",
                    }
                    tmp = text
                    for old, new in replacements.items():
                        if old in tmp.lower() and random.random() < 0.7:
                            tmp = tmp.replace(old, new)
                            break
                    augmented_text = tmp

            augmented_rows.append({
                'text': augmented_text,
                'intent': intent
            })

    if augmented_rows:
        augmented_df = pd.DataFrame(augmented_rows)
        df = pd.concat([df, augmented_df], ignore_index=True)
        print(f"\nTotal augmented samples added: {len(augmented_rows)}")
        print(f"New total: {len(df)} samples")
    else:
        print("No augmentation performed (all selected intents have >= min_samples samples).")

    return df

# Note: we now call augmentation *after* creating the train/val/test split
# so that validation and test metrics are computed only on original data.

## 6. Data Preprocessing

In [64]:
# Keep a copy of the original (non-augmented) data
base_df = df.copy().reset_index(drop=True)

print(f"Total original samples: {len(base_df)}")
print(f"Unique intents: {base_df['intent'].nunique()}")

Total original samples: 694
Unique intents: 5


In [65]:
# Improved train/val/test split that keeps eval data strictly original

def stratified_split_with_min_samples_df(df, test_size=0.2, val_size=0.1, min_test_samples=2):
    """Stratified split on a DataFrame.

    Splits `df` into train/val/test, ensuring that the test set only
    contains original samples (no augmentation) and has at least a
    minimum number of examples per class.
    """
    texts = df['text'].tolist()
    labels = [label2id[intent] for intent in df['intent'].tolist()]

    # First split: train vs temp (70% train, 30% temp)
    train_texts, temp_texts, train_labels, temp_labels = train_test_split(
        texts, labels, test_size=0.3, random_state=SEED, stratify=labels
    )

    # Second split: validation vs test (50% each of temp = 10% each of total)
    val_texts, test_texts, val_labels, test_labels = train_test_split(
        temp_texts, temp_labels, test_size=0.5, random_state=SEED, stratify=temp_labels
    )

    # Verify minimum samples in test set
    test_label_counts = Counter(test_labels)
    min_test_count = min(test_label_counts.values()) if test_label_counts else 0

    print(f"Train set (original only): {len(train_texts)} samples")
    print(f"Validation set (original only): {len(val_texts)} samples")
    print(f"Test set (original only): {len(test_texts)} samples")
    print(f"Minimum samples per class in test: {min_test_count}")

    # Rebuild DataFrames for convenience
    train_df = pd.DataFrame({"text": train_texts, "intent": [id2label[l] for l in train_labels]})
    val_df = pd.DataFrame({"text": val_texts, "intent": [id2label[l] for l in val_labels]})
    test_df = pd.DataFrame({"text": test_texts, "intent": [id2label[l] for l in test_labels]})

    return train_df, val_df, test_df


# Perform split on the *original* data only
train_df, val_df, test_df = stratified_split_with_min_samples_df(
    base_df, test_size=0.2, val_size=0.1, min_test_samples=2
)

print("\nTest-set intent distribution:")
print(test_df["intent"].value_counts())

Train set (original only): 485 samples
Validation set (original only): 104 samples
Test set (original only): 105 samples
Minimum samples per class in test: 5

Test-set intent distribution:
intent
inquiry                42
info                   26
transaction_confirm    22
unknown                10
greeting                5
Name: count, dtype: int64


In [66]:
pass

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import time

true_labels = test_df["intent"].tolist()
pred_labels = []

for idx, text in enumerate(test_df["text"].tolist()):
    pred = classify_intent(text, use_fewshot=True)
    pred_labels.append(pred)

    if (idx + 1) % 10 == 0:
        print(f"Processed {idx+1}/{len(test_df)}")

    time.sleep(0.5)

print("Accuracy:", accuracy_score(true_labels, pred_labels))
print("\nClassification Report:\n", classification_report(true_labels, pred_labels, labels=INTENT_LABELS))
print("\nConfusion Matrix:\n", confusion_matrix(true_labels, pred_labels, labels=INTENT_LABELS))

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

## 7. Model Setup

In [ ]:
pass

In [ ]:
pass

## 8. Training Setup

In [ ]:
pass

In [ ]:
pass

## 9. Training

In [ ]:
pass

In [ ]:
pass

## 10. Evaluation

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

## 11. Save Model

In [ ]:
pass

In [ ]:
pass

## 12. Inference Example

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

pass